In [1]:
import io
import threading
import time
import traceback

import numpy as np
import ipywidgets as w
from IPython.display import display
from matplotlib.figure import Figure
from pynq import Overlay, MMIO, allocate
from pynq.lib.iic import AxiIIC

ol = Overlay("bits/dtmf.bit")
gen_ip = ol.ip_dict["dtmf_up_wrapper_0"]
i2s_ip = ol.ip_dict["i2s_transmitter_0"]
regs = MMIO(gen_ip["phys_addr"], gen_ip["addr_range"])
i2s = MMIO(i2s_ip["phys_addr"], i2s_ip["addr_range"])
iic = AxiIIC(ol.ip_dict["axi_iic_0"])
dma = ol.axi_dma_0

CTRL, B1_1, Y1_1, B1_2, Y1_2, LEN = 0x0C, 0x10, 0x14, 0x18, 0x1C, 0x20
I2S_CTRL, I2S_TIMING = 0x08, 0x20
S2MM_DMACR, S2MM_DMASR = 0x30, 0x34
CODEC = 0x1A

MCLK = 12.286e6
SCLK_DIV = 4
FS = MCLK / (SCLK_DIV * 64)
FRAC = 13
N = 2048
AMP0 = 0.75
VOL0 = 0x65
DUR_MS = 200
DMA_TIMEOUT = 1.0

ROWS = [697, 770, 852, 941]
COLS = [1209, 1336, 1477]
KEYS = [["1", "2", "3"], ["4", "5", "6"], ["7", "8", "9"], ["*", "0", "#"]]
TONES = {KEYS[r][c]: (ROWS[r], COLS[c]) for r in range(4) for c in range(3)}

buf = allocate(shape=(N,), dtype=np.int16)

In [2]:
data = np.zeros(N, dtype=np.int16)
last_key = "-"
plot_lock = threading.Lock()

view = w.IntSlider(value=512, min=64, max=N, step=64, description="Window size", continuous_update=False, style={"description_width": "110px"}, layout=w.Layout(width="520px"))
image = w.Image(format="png")

fig = Figure(figsize=(9, 3), dpi=80)
ax = fig.add_subplot()
line, = ax.plot([], [], lw=1.2)
ax.set_xlabel("n")
ax.set_ylabel("Amp")
ax.grid(True, alpha=0.3)
fig.tight_layout()


def refresh():
    with plot_lock:
        n = min(view.value, len(data))
        y = data[:n] / 2**FRAC
        line.set_data(np.arange(n), y)
        ax.set_xlim(0, max(n - 1, 1))
        top = max(1.1, 1.1 * np.abs(y).max()) if n else 1.1
        ax.set_ylim(-top, top)
        ax.set_title(f"Key {last_key}")
        png = io.BytesIO()
        fig.savefig(png, format="png")
        image.value = png.getvalue()

In [3]:
lock = threading.RLock()
token = 0

status = w.Label(value="Ready")
err_box = w.Textarea(rows=4, disabled=True, layout=w.Layout(width="600px", display="none"))


def set_status(text, err=None):
    status.value = text
    err_box.value = err or ""
    err_box.layout.display = "block" if err else "none"

def _coeffs(w_rad, amp):
    b1 = int(round(2 * np.cos(w_rad) * 2**FRAC))
    w_q = np.arccos(b1 / 2**(FRAC + 1))
    y1 = int(np.clip(round(amp * np.sin(w_q) * 2**FRAC), -32768, 32767))
    return b1, y1

def _wait(done, timeout, what):
    t0 = time.monotonic()
    while not done():
        if time.monotonic() - t0 > timeout:
            raise TimeoutError(what)
        time.sleep(0.0005)

def _reset_dma():
    dma.mmio.write(S2MM_DMACR, 0x4)
    _wait(lambda: not dma.mmio.read(S2MM_DMACR) & 0x4, 1.0, "DMA reset")
    dma.recvchannel.start()

def _codec(reg, val):
    iic.send(CODEC, bytes([(reg << 1) | (val >> 8), val & 0xFF]), 2)
    time.sleep(0.005)

def set_volume(v):
    _codec(0x02, 0x100 | int(v))
    _codec(0x03, 0x100 | int(v))

def init_audio():
    regs.write(CTRL, 0)
    i2s.write(I2S_CTRL, 0)
    i2s.write(I2S_TIMING, SCLK_DIV)
    i2s.write(I2S_CTRL, 1)
    _codec(0x0F, 0x000)
    _codec(0x06, 0x030)
    _codec(0x04, 0x010)
    _codec(0x05, 0x000)
    _codec(0x07, 0x002)
    _codec(0x08, 0x000)
    set_volume(VOL0)
    _codec(0x09, 0x001)
    _codec(0x06, 0x020)

def _apply(key, a):
    f_1, f_2 = TONES[key]
    b_1, y_1 = _coeffs(2 * np.pi * f_1 / FS, a)
    b_2, y_2 = _coeffs(2 * np.pi * f_2 / FS, a)
    regs.write(CTRL, 0)
    regs.read(CTRL)
    _reset_dma()
    regs.write(B1_1, b_1 & 0xFFFF)
    regs.write(Y1_1, y_1 & 0xFFFF)
    regs.write(B1_2, b_2 & 0xFFFF)
    regs.write(Y1_2, y_2 & 0xFFFF)
    regs.write(LEN, N)

def _capture_once():
    def finished():
        sr = dma.mmio.read(S2MM_DMASR)
        if sr & 0x70:
            raise RuntimeError(f"DMA error, DMASR=0x{sr:08x}")
        return sr & 0x1000

    dma.mmio.write(S2MM_DMASR, 0x1000)
    dma.recvchannel.transfer(buf)
    regs.write(CTRL, 1)
    _wait(finished, DMA_TIMEOUT, "DMA transfer timeout")
    buf.invalidate()
    return np.array(buf)

def play_worker(key, a, ms):
    global data, last_key, token
    t0 = time.monotonic()
    try:
        with lock:
            token += 1
            my = token
            _apply(key, a)
            data = _capture_once()
            last_key = key
        set_status(f"Playing {key}")
        refresh()
        time.sleep(max(ms / 1000 - (time.monotonic() - t0), 0))
        with lock:
            if token == my:
                regs.write(CTRL, 0)
                set_status("Ready")
    except Exception:
        set_status("Error: playback stopped", traceback.format_exc())
        try:
            with lock:
                regs.write(CTRL, 0)
        except Exception:
            pass

def play(key, a, ms):
    threading.Thread(target=play_worker, args=(key, a, ms), daemon=True).start()

In [ ]:
def safe(fn, *args):
    try:
        fn(*args)
    except Exception:
        set_status("Error", traceback.format_exc())


def make_key(k):
    b = w.Button(description=k, layout=w.Layout(width="60px", height="50px"))
    b.on_click(lambda _: safe(play, k, AMP0, DUR_MS))
    return b


pad = w.GridBox([make_key(k) for row in KEYS for k in row], layout=w.Layout(grid_template_columns="repeat(3, 60px)"))

view.observe(lambda c: refresh(), "value")

safe(init_audio)
refresh()

display(w.VBox([
    pad,
    view,
    err_box,
    image,
]))